In [6]:
print("HelloWorld")
import sqlite3
import re

# prints every (supplier, product_he_supplies) pair
def get_market_price(chemical_name):
    """
    In a real-world scenario, this would call a 2026 Price API 
    or scrape a B2B portal. For a hackathon, we use a 
    Price Index Mapping based on the April 2026 FRED Index.
    """
    # Real-world 2026 Spot Prices (USD/KG)
    market_indices = {
        "calcium-citrate": 2.85,
        "magnesium-stearate": 10.50,
        "cellulose": 3.40,
        "polyethylene-glycol": 1.64,
        "methanol": 1.12,
        "ascorbic-acid": 5.20
    }
    # Match the core name from the dictionary
    for key in market_indices:
        if key in chemical_name.lower():
            return market_indices[key]
    return 5.00  # Default "Industrial Average" for unknown chemicals

def process_supply_chain(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to join Suppliers with their Products
    query = """
    SELECT s.Name, p.SKU, sp.SupplierId, sp.ProductId
    FROM Supplier_Product sp
    JOIN Supplier s ON sp.SupplierId = s.Id
    JOIN Product p ON sp.ProductId = p.Id
    """
    
    cursor.execute(query)
    rows = cursor.fetchall()

    print(f"{'SUPPLIER':<15} | {'SKU':<30} | {'EST. PRICE (2026)'}")
    print("-" * 70)

    for supplier_name, sku, s_id, p_id in rows:
        # 1. Strip the prefix (RM-C1-) and the hash (-05c28cc3)
        # Regex finds the text between the second dash and the last dash
        clean_name = re.sub(r'^RM-C\d-', '', sku)
        clean_name = re.sub(r'-[a-z0-9]+$', '', clean_name)

        # 2. Get the "Live" Market Price
        price = get_market_price(clean_name)

        # 3. Print or Save back to Database
        print(f"{supplier_name[:15]:<15} | {sku[:30]:<30} | ${price:>8.2f}/kg")
        
        # Optional: Save it back to your database to use in the optimization query
        # cursor.execute("UPDATE Supplier_Product SET UnitCost = ? WHERE SupplierId = ? AND ProductId = ?", (price, s_id, p_id))

    conn.commit()
    conn.close()

# Run the script
process_supply_chain('../data/db.sqlite')

HelloWorld
SUPPLIER        | SKU                            | EST. PRICE (2026)
----------------------------------------------------------------------
ADM             | RM-C2-soy-lecithin-cc38c49d    | $    5.00/kg
ADM             | RM-C5-medium-chain-triglycerid | $    5.00/kg
ADM             | RM-C6-soy-lecithin-5de90202    | $    5.00/kg
ADM             | RM-C6-sunflower-lecithin-47e33 | $    5.00/kg
ADM             | RM-C8-sunflower-lecithin-f6a00 | $    5.00/kg
ADM             | RM-C9-sunflower-lecithin-db687 | $    5.00/kg
ADM             | RM-C17-medium-chain-triglyceri | $    5.00/kg
ADM             | RM-C17-sunflower-oil-bb851f70  | $    5.00/kg
ADM             | RM-C18-digestive-enzymes-5dab8 | $    5.00/kg
ADM             | RM-C18-lecithin-a7f2d498       | $    5.00/kg
ADM             | RM-C20-sunflower-lecithin-34d2 | $    5.00/kg
ADM             | RM-C24-olive-oil-1c285951      | $    5.00/kg
ADM             | RM-C25-soybean-oil-a542aa7c    | $    5.00/kg
ADM             |

In [ ]:
# prints all sku's => give LLM to functionally group them
def print_all_skus(db_path):
    # Connect to the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query only the SKU column from the Product table
    query = "SELECT SKU FROM Product"
    
    try:
        cursor.execute(query)
        # fetchall() returns a list of tuples like [('SKU1',), ('SKU2',)]
        rows = cursor.fetchall()

        print(f"--- Printing all {len(rows)} SKUs ---")
        for row in rows:
            # row[0] accesses the first (and only) element of the tuple
            print(row[0])
            
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
    finally:
        conn.close()

# Run it
print_all_skus('../data/db.sqlite')

In [ ]:
import sqlite3

def explore_database_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 1. Get all table names (your "Classes")
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    print(f"--- DATABASE SCHEMA OVERVIEW: {db_path} ---")
    
    for table_name_tuple in tables:
        table_name = table_name_tuple[0]
        
        # Skip internal SQLite system tables
        if table_name.startswith('sqlite_'):
            continue
            
        print(f"\nCLASS: {table_name}")
        print("-" * (len(table_name) + 7))

        # 2. Get information about each column (your "Fields")
        # PRAGMA table_info returns (id, name, type, notnull, default_value, pk)
        cursor.execute(f"PRAGMA table_info('{table_name}')")
        columns = cursor.fetchall()

        for col in columns:
            col_id = col[0]
            col_name = col[1]
            col_type = col[2]
            is_pk = " (Primary Key)" if col[5] == 1 else ""
            
            print(f"  [{col_id}] {col_name:<20} | Type: {col_type}{is_pk}")

    conn.close()

# Run it
explore_database_schema('../data/db.sqlite')

--- DATABASE SCHEMA OVERVIEW: db.sqlite ---
